<a href="https://colab.research.google.com/github/ron360/Deep-Learning/blob/main/112306073_%E8%B3%87%E7%AE%A1%E4%B8%89_%E6%9D%8E%E5%B2%B3%E8%9E%8D_%E6%89%93%E9%80%A0RAG%E7%B3%BB%E7%B5%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#我所使用的知識來源是論語原文以及注釋，設計出的RAG是希望能夠根據論語的內容回答使用者提問的行為或道德準則，以作為參考

https://docs.google.com/document/d/1ZGgRzOPMQ72KS9ZkZ1qO2prdODyBQ3vL1l5L7p1sKcQ/edit?usp=drive_link

這是我所使用的知識來源

### 0. 讀入你打造好的 vector dataset

In [ ]:
!pip -q install gdown

In [ ]:
GDRIVE_PUBLIC_URL = "https://drive.google.com/file/d/1dmUWUb0fweLoNSTInS6OB9jmovHZfb3R/view?usp=drive_link"

In [ ]:
!gdown --fuzzy -O faiss_db.zip "{GDRIVE_PUBLIC_URL}"

In [ ]:
!unzip faiss_db.zip

### 1. 安裝並引入必要套件

In [ ]:
!pip install -U langchain langchain-community faiss-cpu transformers sentence-transformers huggingface_hub
!pip -q install "aisuite[all]"

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.chat_models import ChatOpenAI

In [ ]:
from openai import OpenAI
import gradio as gr

### 2. 自訂 E5 embedding 類別

In [ ]:
import os
from google.colab import userdata

In [ ]:
hf_token = userdata.get('HuggingFace')

In [ ]:
from huggingface_hub import login
login(token=hf_token)

In [ ]:
class EmbeddingGemmaEmbeddings(HuggingFaceEmbeddings):
    def __init__(self, **kwargs):
        super().__init__(
            model_name="google/embeddinggemma-300m",
            encode_kwargs={"normalize_embeddings": True},
            **kwargs
        )

    def embed_documents(self, texts):
        # 你也可以把 "none" 改成真實標題（檔名/章節名），效果會更穩
        texts = [f"title: none | text: {t}" for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        # 官方檢索建議前綴
        return super().embed_query(f"task: search result | query: {text}")

### 3. 載入 `faiss_db`

In [ ]:
embedding_model = EmbeddingGemmaEmbeddings()
vectorstore = FAISS.load_local(
    "faiss_db",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

### 4. 設定好我們要的 LLM

如之前, 我們會用 OpenAI API。這裡使用 Groq 服務, 可改成你要的服務。

In [ ]:
import aisuite as ai

In [ ]:
api_key = userdata.get('Groq')

In [ ]:
os.environ['GROQ_API_KEY']=api_key

這裡的模型和 `base_url` 是用 Groq, 如果用其他服務請自行修改。

In [ ]:
model = "groq:openai/gpt-oss-120b"
#base_url="https://api.groq.com/openai/v1"

In [ ]:
client = ai.Client()

### 5. `prompt` 設計

In [ ]:
system_prompt = "你是政大的 AI 論語教授，請根據資料來回應學生的問題。請完整且合理的說明論述。請用台灣習慣的中文回應。"  # 更改Prompt 調整人設

prompt_template = """
根據下列資料：
{retrieved_chunks}

回答使用者的問題：{question}

請根據資料內容回覆，若資料不足請告訴同學可以查詢原文片段再進行提問。
"""

### 6. 使用 RAG 來回應

搜尋與使用者問題相關的資訊，根據我們的 prompt 樣版去讓 LLM 回應。

In [ ]:
chat_history = []

def chat_with_rag(user_input):
    global chat_history
    # 取回相關資料
    docs = retriever.invoke(user_input)
    retrieved_chunks = "\n\n".join([doc.page_content for doc in docs])

    # 將自定 prompt 套入格式
    final_prompt = prompt_template.format(retrieved_chunks=retrieved_chunks, question=user_input)

    # 用 AI Suite 呼叫語言模型
    response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": final_prompt},
    ]
    )
    answer = response.choices[0].message.content

    chat_history.append((user_input, answer))
    return answer

### 7. 用 Gradio 打造 Web App

In [ ]:
with gr.Blocks() as demo:     # 調整Gradio介面
    gr.Markdown("# 🎓 AI 論語教授")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(placeholder="請輸入你的問題...")

    def respond(message, chat_history_local):
        response = chat_with_rag(message)
        chat_history_local.append((message, response))
        return "", chat_history_local

    msg.submit(respond, [msg, chatbot], [msg, chatbot])

demo.launch(debug=True)